In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin

# =====================
# Load data
# =====================
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

X_train = train["Versuri"].astype(str)
X_test  = test["Versuri"].astype(str)

y = train["Autor"]
test_id = test["Id"]

# =====================
# Custom structural features
# =====================
class PoetryStructure(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        feats = []

        for text in X:
            lines = text.split("\n")
            lines = lines[:4] + [""] * (4 - len(lines))

            line_lengths = [len(l) for l in lines]
            word_lengths = [np.mean([len(w) for w in l.split()]) if l.split() else 0 for l in lines]

            punctuation = sum(l.count(",") + l.count(";") + l.count(":") for l in lines)
            dots = sum(l.count(".") for l in lines)

            rhyme = [l.strip()[-3:] if len(l.strip()) >= 3 else "" for l in lines]
            rhyme_match = sum(rhyme[i] == rhyme[j] for i in range(4) for j in range(i+1,4))

            feats.append([
                np.mean(line_lengths),
                np.std(line_lengths),
                np.mean(word_lengths),
                punctuation,
                dots,
                rhyme_match
            ])

        return np.array(feats)

# =====================
# Char-level style
# =====================
char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 6),
    min_df=2,
    sublinear_tf=True
)

# =====================
# Combine features
# =====================
vectorizer = FeatureUnion([
    ("char", char_vec),
    ("struct", PoetryStructure())
])

X_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# =====================
# Encode labels
# =====================
le = LabelEncoder()
y_enc = le.fit_transform(y)

# =====================
# Model
# =====================
model = LogisticRegression(
    solver="saga",
    C=4,
    max_iter=5000,
    n_jobs=-1
)

model.fit(X_vec, y_enc)

# =====================
# Predict
# =====================
pred = model.predict(X_test_vec)
pred_labels = le.inverse_transform(pred)

submission = pd.DataFrame({
    "Id": test_id,
    "Autor": pred_labels
})

submission.to_csv("submission.csv", index=False)


KeyboardInterrupt: 